# Nonograms with ShadowProver
### Colab quick-start (Python 3.13)

This notebook writes the rules of a nonogram as first-order logic and solves the puzzle by theorem proving. A cell is only filled in or crossed out once the prover has proved its value, so every step of a solution has a justification.

**Requirements**
1. A Colab runtime on **Python 3.13** (`!python --version`). The wheel is version-locked to 3.13.
2. Higher-order E-prover (built below).

## 1. Check the Python version

In [1]:
!python --version   # expect Python 3.13.x

Python 3.14.5


## 2. Install shadowprover (from S3)

In [2]:
!pip install -q https://shadowprover-spectra.s3.amazonaws.com/py313/shadowprover-1.2.59-py3-none-any.whl
!pip install -q python-Levenshtein


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Build higher-order E-prover
Pinned to the commit this notebook was tested against. `make` takes a few minutes.

In [3]:
!git clone -q https://github.com/eprover/eprover
!cd eprover && git checkout -q 543cb16baabe34ca9d55374dd840f76b30048bb3 && ./configure --enable-ho > /dev/null && make rebuild > /dev/null 2>&1
!./eprover/PROVER/eprover-ho --version

The system cannot find the path specified.
'.' is not recognized as an internal or external command,
operable program or batch file.


## 4. Point shadowprover at E-prover
This has to run before anything imports the prover.

In [4]:
import os
os.environ['EPROVER_HOME'] = '/content/eprover/'

## 5. The rules of a nonogram (yours to write)
`line_rules` below returns the rules for one row or column. Each rule is a string in ShadowProver's s-expression syntax. The solver translates each one to first-order logic and gives it to E as an axiom.

| Write | Meaning |
|---|---|
| `(Black r0 c3)` | An atom: the predicate, then its arguments |
| `(not P)` | not P |
| `(and P Q ...)`, `(or P Q ...)` | P and Q, P or Q (any number of parts) |
| `(if P Q)` | P implies Q |
| `(iff P Q)` | P if and only if Q |
| `(forall (x y) P)`, `(exists (x) P)` | Quantifiers. Variables are the names listed after the quantifier. |
| `(= a b)` | a and b are the same thing |

Things that silently go wrong:
- **Arguments must be names**, such as `c3` or `p2`. A bare number like `(Black r0 3)` makes the reader raise a TypeError.
- **`(or )` with nothing inside** raises a TypeError. Use `any_of`, which also turns a one-item `or` into just that item.
- **Unknown operators are not errors.** `implies`, `xor` and similar become ordinary predicates, so the rule says nothing. Use `if`.
- **Names ignore case.** `Black` and `black` are the same predicate.
- **Different names are not assumed different.** If a rule relies on `p0` and `p1` being different positions, also add `(not (= p0 p1))`.
- **Write rules as formulas**, not as ShadowProver inference schemas (`read_inference_system`). In version 1.2.59, those schemas don't fire through the prover.

What the solver needs from your rules:
- The rules plus the known cells of a line should prove a cell `(Black ...)` or `(not (Black ...))` whenever the clue forces it.
- When the known cells can't fit the clue, the rules should prove a contradiction. The "assume and check" step depends on this.

Rules that are too weak make the solver guess more. Rules that are wrong can make it report "no solution", but they can't produce a wrong grid, because the final grid is checked against the clues directly.

In [ ]:
from itertools import combinations, groupby

ABSURD = "(and (Absurd x) (not (Absurd x)))"


def black(row, col):
    # Names rather than bare integers: the reader raises TypeError on numeric arguments.
    return f"(Black r{row} c{col})"


def any_of(atoms):
    # The reader raises TypeError on an empty "(or )", and a one-item "or" is just that item.
    return atoms[0] if len(atoms) == 1 else f"(or {' '.join(atoms)})"


def line_rules(line, clue, cells):
    """Your nonogram rules for one row or column, as a list of formula strings.

    line:  the line's name, "row3" or "col0"
    clue:  the block lengths, such as [1, 2], or [] for an empty line
    cells: the line's cell atoms in order, such as ["(Black r3 c0)", "(Black r3 c1)", ...]
    """
    rules = []

    def start(block, position):
        # "(Start row3 b0 p2)" means "block 0 of row3 starts at cell 2".
        return f"(Start {line} b{block} p{position})"

    # Example rule: a block paints the cells it covers black.
    # If a block of length L starts at cell p, then cells p .. p+L-1 are black.
    for b, length in enumerate(clue):
        for p in range(len(cells) - length + 1):  # every start where the block fits in the line
            for offset in range(length):
                rules.append(f"(if {start(b, p)} {cells[p + offset]})")

    # TODO: your other rules go here.

    return rules


# The rules for a 5-cell row with clue [3]
for rule in line_rules("row0", [3], [black(0, c) for c in range(5)]):
    print(rule)

## 6. Asking the prover
`fol_prove` gives E only **0.1 seconds** and reports a timeout as `False`, so a harder theorem looks unprovable. `ask` keeps ShadowProver's parser and translation to TPTP, but runs E with a real time budget and separates "disproved" from "ran out of time".

Treat "no proof found" as "don't know yet", never as "false".

In [6]:
import re
import subprocess
from functools import lru_cache

from shadowprover.fol.fol_prover import fol_prove, to_tptp_formula
from shadowprover.syntax.reader import r

PROOF_SECONDS = 10
stats = {"proofs": 0, "gave_up": 0, "guesses": 0}


@lru_cache(maxsize=None)
def tptp(formula):
    return to_tptp_formula(r(formula))


def ask(assumptions, goal, seconds=None):
    """Return 'proved', 'disproved' or 'unknown'."""
    lines = [f"fof(a{i}, axiom, {tptp(a)})." for i, a in enumerate(assumptions)]
    lines.append(f"fof(goal, conjecture, {tptp(goal)}).")
    stats["proofs"] += 1
    done = subprocess.run(
        [os.environ["EPROVER_HOME"] + "PROVER/eprover-ho", "--auto", f"--cpu-limit={seconds or PROOF_SECONDS}", "-s"],
        input="\n".join(lines) + "\n",
        capture_output=True,
        text=True,
    )
    status = re.search(r"SZS status (\w+)", done.stdout)
    status = status.group(1) if status else "None"
    # E says ContradictoryAxioms rather than Theorem when the assumptions clash.
    if status in ("Theorem", "ContradictoryAxioms"):
        return "proved"
    if status in ("CounterSatisfiable", "Satisfiable"):
        return "disproved"
    stats["gave_up"] += 1
    return "unknown"


# Pigeonhole: 8 pigeons cannot fit in 7 holes. A real theorem, but it takes E more than 0.1s.
holes = 7
pigeonhole = ["(or %s)" % " ".join(f"(In a{p} h{h})" for h in range(holes)) for p in range(holes + 1)]
pigeonhole += [
    f"(not (and (In a{p} h{h}) (In a{q} h{h})))"
    for h in range(holes)
    for p, q in combinations(range(holes + 1), 2)
]
print("fol_prove:", fol_prove({r(a) for a in pigeonhole}, r(ABSURD))[0])
print("ask:      ", ask(pigeonhole, ABSURD))

ImportError: DLL load failed while importing pyarmor_runtime: The specified module could not be found.

## 7. The solver
1. **Prove what one line forces.** For each unknown cell, ask whether its line's rules plus the known cells prove it black or white. A new cell can unlock the line crossing it, so that line gets rechecked.
2. **When stuck, assume.** Pick an unknown cell, assume it is black and keep going. A contradiction proves the assumption wrong, so the cell is white. Some puzzles need this, because no single line settles them.

If the prover runs out of time on a query, the cell just stays unknown and costs more search. It can't produce a wrong cell, and the finished grid is also checked against the clues directly.

In [ ]:
class Puzzle:
    def __init__(self, row_clues, col_clues):
        self.row_clues, self.col_clues = row_clues, col_clues
        self.height, self.width = len(row_clues), len(col_clues)
        self.lines = {}
        for i, clue in enumerate(row_clues):
            coords = [(i, c) for c in range(self.width)]
            self.lines[f"row{i}"] = (coords, line_rules(f"row{i}", clue, [black(*x) for x in coords]))
        for j, clue in enumerate(col_clues):
            coords = [(c, j) for c in range(self.height)]
            self.lines[f"col{j}"] = (coords, line_rules(f"col{j}", clue, [black(*x) for x in coords]))


def propagate(puzzle, known, todo, trace):
    """Prove every cell that follows from one line at a time. False means a contradiction."""
    todo = set(todo)
    while todo:
        name = todo.pop()
        coords, rules = puzzle.lines[name]
        facts = [black(*x) if known[x] else f"(not {black(*x)})" for x in coords if x in known]
        if ask(rules + facts, ABSURD) == "proved":
            trace.append(f"{name} is contradictory, so the last assumption was wrong")
            return False
        for x in coords:
            if x in known:
                continue
            for value, goal in ((True, black(*x)), (False, f"(not {black(*x)})")):
                if ask(rules + facts, goal) == "proved":
                    known[x] = value
                    facts.append(goal)
                    trace.append(f"{name} proves r{x[0]} c{x[1]} is {'black' if value else 'white'}")
                    todo.add(f"col{x[1]}" if name.startswith("row") else f"row{x[0]}")
                    break
    return True


def runs(values):
    return [sum(1 for _ in group) for value, group in groupby(values) if value]


def matches_clues(puzzle, known):
    rows, cols = range(puzzle.height), range(puzzle.width)
    return all(runs(known[i, j] for j in cols) == puzzle.row_clues[i] for i in rows) and all(
        runs(known[i, j] for i in rows) == puzzle.col_clues[j] for j in cols
    )


def solve(puzzle, trace=None):
    """Return {(row, col): is_black}, or None if the clues have no solution."""
    trace = [] if trace is None else trace
    stats.update(proofs=0, gave_up=0, guesses=0)

    def search(known, todo):
        if not propagate(puzzle, known, todo, trace):
            return None
        unknown = [
            (i, j)
            for i in range(puzzle.height)
            for j in range(puzzle.width)
            if (i, j) not in known
        ]
        if not unknown:
            return known if matches_clues(puzzle, known) else None
        cell = unknown[0]
        stats["guesses"] += 1
        for value in (True, False):
            trace.append(f"stuck: assume r{cell[0]} c{cell[1]} is {'black' if value else 'white'}")
            attempt = dict(known)
            attempt[cell] = value
            result = search(attempt, {f"row{cell[0]}", f"col{cell[1]}"})
            if result is not None:
                return result
        return None

    return search({}, set(puzzle.lines))


def show(puzzle, solution):
    for i in range(puzzle.height):
        print("".join("#" if solution[i, j] else "." for j in range(puzzle.width)))

## 8. Solve a 5x5

In [ ]:
import time

puzzle = Puzzle(
    row_clues=[[1, 2], [1, 2], [1, 2], [1, 1], [2, 2]],
    col_clues=[[2, 1], [3], [], [3, 1], [5]],
)
trace = []
started = time.time()
solution = solve(puzzle, trace)
print(f"{time.time() - started:.1f}s, {stats}\n")
show(puzzle, solution)
print("\nFirst steps of the reasoning:")
print("\n".join(trace[:8]))

## 9. A puzzle that needs assumptions
No single line settles this one, so the solver has to assume a cell and follow the consequences.

In [ ]:
puzzle = Puzzle(
    row_clues=[[2, 1], [1, 1, 1], [1, 1], [1, 1, 1], [2], [3]],
    col_clues=[[1, 1], [1, 1, 1], [2, 1], [1, 1], [1, 1], [3, 1]],
)
trace = []
started = time.time()
solution = solve(puzzle, trace)
print(f"{time.time() - started:.1f}s, {stats}\n")
show(puzzle, solution)
print("\nThe assumptions it made:")
print("\n".join(step for step in trace if step.startswith("stuck") or "contradictory" in step))

## 10. Try your own
Type in the clues from [puzzle-nonograms.com](https://www.puzzle-nonograms.com/), top to bottom and left to right. Use `[]` for an empty line.

In [ ]:
my_puzzle = Puzzle(
    row_clues=[[1], [3], [1]],
    col_clues=[[1], [3], [1]],
)
my_solution = solve(my_puzzle)
if my_solution is None:
    print("These clues have no solution. Check for a typo.")
else:
    print(stats)
    show(my_puzzle, my_solution)